# Falcon Noise Analysis

The noise in the Falcon-style (dual-complement, single-disc) airborne gravity gradiometer is well characterised by the difference noise (Dransfield REF). Thus the noise in each component is half the standard deviation of the difference between complements):

$$N_{NE}=0.5\,\sigma\left(A_{NE}-B_{NE}\right)$$
$$N_{UV}=0.5\,\sigma\left(A_{UV}-B_{UV}\right)$$

It is also useful to check for excess high frequency signal, and to check for any demodulation phase error as described below.

We will use the Canobie Falcon data for these examples so ensure you have run the `Prepare_XYZ` notebook first so that these data are prepared for review.

___

Import the required modules, set the path to the geowhizz files and list the channels.

In [3]:
from pathlib import Path

import AirGravQC as qc

In [4]:
data_root = r'./CanobieData/'
canobieHDF_file = Path(data_root + r'Canobie.hdf5')

plan_root = data_root
canobieHDF_plan = Path(plan_root + r'CanobiePlan.hdf5')

In [5]:
if not canobieHDF_file.exists():
    %run ./Prepare_CanobieData.ipynb

In [6]:
qc.reportChannels(canobieHDF_file)

Whizz Version 1.0

33 channels:


33 channels:
 ['ANE_TC_2p67', 'AUV_TC_2p67', 'BNE_TC_2p67', 'BUV_TC_2p67', 'Bearing', 'CLEARANCE', 'DTM', 'Date', 'EASTING', 'FIDUCIAL', 'FLIGHT', 'GDD_Fourier_2p67', 'GNE_Fourier_2p67', 'GUV_Fourier_2p67', 'HDOP', 'HEIGHT', 'JOB_ID', 'LATITUDE', 'LINE', 'LONGITUDE', 'NORTHING', 'Noise_NE', 'Noise_UV', 'NumSats', 'PDOP', 'TURBULENCE', 'T_DD', 'T_NE', 'T_UV', 'Time_1980', 'Time_Day', 'VDOP', 'gD_Fourier_2p67']


In [7]:
seen = set()

In [10]:
seen.add('123')
seen

{'123'}

In [22]:
def checkDuplicateLines(whizzFile):
    """
    Check the flight-lines in whizzfile and report any duplicate group names.

    Parameters
    ----------
    whizzFile : String or pathlib Path
        Name of a HDF5 Whizz file, including path and extension.

    Returns
    -------
    None.

    """
    filename = str(whizzFile)

    with h5py.File(filename, 'r+') as f:
        # create all the data structure ready for the datasets
        g = f[groupName]['Lines']
        flightlines = list(g.keys())
    # `set` removes non-unique elements. 
    seen = set()

    # Check for duplicates
    unique = True
    for x in flightlines:
        if x in seen:
            unique = False
            print(f'WARNING - duplicate flight-line {x} in {filename}.')
            break
        seen.add(x)            


In [23]:
import h5py
import AirGravQC.config as config

groupName = config.groupName
checkDuplicateLines(canobieHDF_file)

In [16]:
fid = h5py.File(canobieHDF_file, 'r+')

In [18]:
g = fid[groupName]['Lines']

In [21]:
list(g.keys())

['100010.000',
 '100020.000',
 '100030.000',
 '100040.000',
 '100050.000',
 '100060.000']

___

The **`diffNoiseVturb`** function plots the difference noise for each survey line against the turbulence for that line. Higher turbulence generally results in higher noise and the plot allows one to decide on a reasonable turbulence limit for the data. In addition, flight-lines with a difference noise that is off the general trend (high noise at low turbulence) should be questioned even if they meet specification.

In this example, we get very low noise estimates and there are no data off-trend.

The usual `error_spec` is $5\,E$ but here we have set it to $2.5\,E$ in order to force an error message, purely for an example. You can see that lines that fail the set specification are labeled on the plot.

In [ ]:
qc.diffNoiseVturb(canobieHDF_file, eNE='Noise_NE', eUV='Noise_UV', turbulence='TURBULENCE',
                  error_spec=2.5, labelLines=True)

___

Any rectification process can down-convert high frequency vibration into the signal band of a sensor. Gravity gradiometers have an intrinsic rectification process via their sensitivity to products of rotational velocity so it is useful to check for excess high frequency signal since it may lead to error in the final data (**REF Sutherland et al**).

The function **`checkHighFreq`** checks for unusually high amplitude, high frequency signal in the raw gradient channels. The input gradient channels are high-pass filtered and a rolling standard deviation is used to find periods of higher amplitudes. High turbulence is sometimes associated with high frequency noise so the turbulence is also plotted for comparison.

Experience with Falcon data suggests that sections of data where `checkHighFreq` finds high frequency noise above $15\,E$ are of concern and ought to be followed up with the service provider.

A plot showing where the high frequency noise occurs along line is shown for all lines that have excess high frequency signal. The Canobie data do not have any high frequency noise problems, so the `noiseLimit` has been artifically set here to $6.5\,E$ just to get a plot for the example.

In [ ]:
qc.checkHighFreq(canobieHDF_file, noiseLimit=6.5, 
                 channels=['ANE_TC_2p67', 'AUV_TC_2p67', 'BNE_TC_2p67', 'BUV_TC_2p67'], 
                 cutoffs=[0.15, 3.6], vertaccel='TURBULENCE', verbose=True, plot_flag=True)

___

There is also a function available to check **demodulation phase**. Because the data are measured on a rotating disk, they must be demodulated. The demodulation phase establishes the angular position of the wheel. An incorrect value leads to mixing of signal between the demodulated components and thus also in all generated gravity gradient components.

Errors are extremely rare but possible. The A and B complements are demodulated independently. If one of them has an incorrect demodulation phase, then the two complements will not be as well correlated with each other as possible. The `checkPhase` function iteratively adjusts the demodulation phase of one complement and reports the correlation as this is varied. The maximum correlation should occur at $0.0$ adjustment.

If any flight-line has its maximum correlation at an absolute phase difference greater than the `tolerance`, then its correlation as a function of phase difference is plotted and the predicted offset phase reported. Noise in the data limits the capability of this algorithm and phase differences less than 3.0 are not reliable.

Here the `tolerance` is set to 0.5 merely so as to generate a plot for the example. In fact, the plot shows clearly that the demodulation phases have led to very good correlation.

In [ ]:
qc.checkPhase(canobieHDF_file, 'ANE_TC_2p67', 'BNE_TC_2p67', tolerance=0.5, lines=[], plot_flag=True)